In [2]:
%load_ext autoreload
%autoreload 2
%pip install catboost

import sys
import time
import numpy as np
sys.path.insert(0, '../src')

from catboost import CatBoostClassifier
from recsys.data import load_ratings, make_positive, temporal_split, user_items
from recsys.metrics import evaluate
from recsys.ranking import FEATURES, add_rank_features, build_candidates, rerank

ratings = load_ratings('../data')
pos = make_positive(ratings)
train, test = temporal_split(pos)
train_core, valid = temporal_split(train)

params = {'factors': 64, 'reg': 100, 'alpha': 10}

# 1. Обучающая таблица ранкера: история train_core, метки из valid
valid_users = list(user_items(valid))
cand = add_rank_features(build_candidates(train_core, valid_users, params, n_candidates=100, target=valid))

# 2. Делим ПОЛЬЗОВАТЕЛЕЙ (не строки!) на обучающих и контрольных для ранней остановки
rng = np.random.default_rng(0)
eval_users = set(rng.choice(valid_users, size=len(valid_users) // 5, replace=False))
is_eval = cand['user_id'].isin(eval_users)
fit_df, eval_df = cand[~is_eval], cand[is_eval]

# 3. Обучаем CatBoost
model = CatBoostClassifier(iterations=500, learning_rate=0.1, depth=6, eval_metric='AUC',
                           random_seed=42, verbose=100, early_stopping_rounds=50)
start = time.time()
model.fit(fit_df[FEATURES], fit_df['label'], eval_set=(eval_df[FEATURES], eval_df['label']))
print(f'обучение: {time.time() - start:.0f} сек, лучшая итерация: {model.get_best_iteration()}')
print('важность признаков:', {f: round(float(v), 1) for f, v in zip(FEATURES, model.get_feature_importance())})

# 4. Финальная проверка: история = весь train, ответы = test (ранкер их не видел)
truth = user_items(test)
test_users = list(truth)
test_cand = add_rank_features(build_candidates(train, test_users, params, n_candidates=100))

base = rerank(test_cand, -test_cand['als_rank'])                              # порядок ALS
ranked = rerank(test_cand, model.predict_proba(test_cand[FEATURES])[:, 1])    # порядок ранкера

for k in [10, 20, 50]:
    print('ALS   ', evaluate(base, truth, k))
    print('Ranker', evaluate(ranked, truth, k))

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


  Using cached catboost-1.2.10-cp314-cp314-win_amd64.whl.metadata (1.5 kB)
  Using cached graphviz-0.21-py3-none-any.whl.metadata (12 kB)
  Using cached matplotlib-3.11.2-cp314-cp314-win_amd64.whl.metadata (80 kB)
  Using cached plotly-7.1.0-py3-none-any.whl.metadata (8.8 kB)
  Using cached contourpy-1.4.0-cp314-cp314-win_amd64.whl.metadata (3.8 kB)
  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
  Using cached fonttools-4.67.0-cp314-cp314-win_amd64.whl.metadata (139 kB)
  Using cached kiwisolver-1.5.1-cp314-cp314-win_amd64.whl.metadata (5.2 kB)
  Using cached pillow-13.0.0-cp314-cp314-win_amd64.whl.metadata (9.3 kB)
  Using cached pyparsing-3.3.3-py3-none-any.whl.metadata (5.9 kB)
Using cached catboost-1.2.10-cp314-cp314-win_amd64.whl (101.7 MB)
Using cached graphviz-0.21-py3-none-any.whl (47 kB)
Using cached matplotlib-3.11.2-cp314-cp314-win_amd64.whl (9.5 MB)
Using cached contourpy-1.4.0-cp314-cp314-win_amd64.whl (240 kB)
Using cached cycler-0.12.1-py3-none-any.whl (


[notice] A new release of pip is available: 25.3 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


0:	test: 0.6522662	best: 0.6522662 (0)	total: 175ms	remaining: 1m 27s
100:	test: 0.6887954	best: 0.6887954 (100)	total: 2.87s	remaining: 11.3s
200:	test: 0.6898409	best: 0.6898846 (154)	total: 5.55s	remaining: 8.26s
Stopped by overfitting detector  (50 iterations wait)

bestTest = 0.6898896221
bestIteration = 201

Shrink model to first 202 iterations.
обучение: 7 сек, лучшая итерация: 201
важность признаков: {'als_rank': 12.0, 'als_score': 37.1, 'knn_score': 15.6, 'knn_rank': 12.5, 'item_pop': 15.1, 'user_activity': 7.8}
ALS    {'recall@10': 0.07592583915550757, 'ndcg@10': 0.10207782303367445}
Ranker {'recall@10': 0.08324416566215603, 'ndcg@10': 0.11193145342573141}
ALS    {'recall@20': 0.1360849706187366, 'ndcg@20': 0.12036701483805519}
Ranker {'recall@20': 0.14503931050328114, 'ndcg@20': 0.12974653436394878}
ALS    {'recall@50': 0.2687120261193667, 'ndcg@50': 0.16718816299628128}
Ranker {'recall@50': 0.2774966508035686, 'ndcg@50': 0.1755555706848771}
